# Retrieval Evaluation

We compare three retrieval strategies against the ground-truth question set generated in `ground-truth-generation.ipynb`, using Hit Rate and MRR:

1. **Keyword search** — TF-IDF via `minsearch.Index` (the current production search in `rag.py`).
2. **Vector search** — cosine similarity over embeddings from a local ONNX MiniLM model (`coffee_assistant/embedder.py`), no API calls.
3. **Hybrid search** — Reciprocal Rank Fusion (RRF) of both.

The winning approach is wired into `rag.py`'s `search()`.

In [1]:
import pandas as pd
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')

In [2]:
df_question.head()

,chunk_id,question
0,604727_0,What are the main steps involved in coffee pro...
1,604727_0,How does the flavor of coffee change with the ...
2,604727_0,In which historical context did the earliest r...
3,604727_0,What are the two most commonly grown types of ...
4,604727_0,What was the approximate global worth of the c...


In [3]:
ground_truth = df_question.to_dict(orient='records')

In [4]:
ground_truth[0]

{'chunk_id': '604727_0',
 'question': 'What are the main steps involved in coffee production from the coffee cherry to brewing?'}

## Metrics: Hit Rate and MRR

`evaluate` runs a search function over every ground-truth question and checks whether the correct `chunk_id` shows up in the results.

In [5]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [6]:
from tqdm.auto import tqdm
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        chunk_id = q['chunk_id']
        results = search_function(q)
        relevance = [d['chunk_id'] == chunk_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

## 1. Keyword search (baseline)

`retrieval.keyword_search()` is the plain TF-IDF search, evaluated in isolation from the hybrid pipeline below — note this is deliberately *not* `rag.search()`, since that now runs hybrid search in production (see the wiring at the bottom of this notebook).

In [7]:
import sys
sys.path.append('..')

from coffee_assistant import rag, retrieval

In [8]:
keyword_results = evaluate(ground_truth, lambda q: retrieval.keyword_search(q['question'], num_results=10))
keyword_results

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.5563492063492064, 'mrr': 0.3384571050642485}

## 2. Vector search

Embeddings are built by `retrieval.get_vector_index()` — every chunk embedded with a local ONNX MiniLM model (`coffee_assistant/embedder.py`, no PyTorch/API calls), rebuilt fresh in memory each run. `retrieval.vector_search()` ranks by cosine similarity (the embeddings are L2-normalized, so a dot product is enough).

In [9]:
vector_results = evaluate(ground_truth, lambda q: retrieval.vector_search(q['question'], 10))
vector_results

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.6333333333333333, 'mrr': 0.4043455530360299}

## 3. Hybrid search (Reciprocal Rank Fusion)

`retrieval.hybrid_search()` merges the keyword and vector result lists by summing `1 / (k + rank)` per method for each `chunk_id` (`retrieval.rrf`), then ranks by the combined score. `k` is a smoothing constant from the RRF literature.

In [10]:
# k=60 here (the RRF literature default) purely to demonstrate the tuning point below —
# retrieval.hybrid_search()'s actual default is k=1, the winner picked further down.
hybrid_results = evaluate(ground_truth, lambda q: retrieval.hybrid_search(q['question'], k=60))
hybrid_results

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.7507936507936508, 'mrr': 0.39130401234568035}

### Tuning the RRF `k` parameter

The common default `k=60` comes from search literature where rankings run into the thousands. Our lists only have 10 candidates each (rank 0–9), so at `k=60` the score `1/(k+rank)` barely differs between rank 0 and rank 9 — RRF ends up mostly rewarding "appears in both lists" and ignoring *where*. A much smaller `k` restores that signal. Sweep a few values to check the effect on Hit Rate and MRR.

In [11]:
for k in [1, 5, 10, 50, 60, 100, 200]:
    result = evaluate(ground_truth,
                      lambda q, k=k: retrieval.hybrid_search(q['question'], k))
    print(f"Hybrid search (k={k}): Hit Rate={result['hit_rate']:.4f}, MRR={result['mrr']:.4f}")

  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=1): Hit Rate=0.7508, MRR=0.4213


  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=5): Hit Rate=0.7508, MRR=0.4001


  0%|          | 0/2520 [00:00<?, ?it/s]

...

Hybrid search (k=60): Hit Rate=0.7508, MRR=0.3913


  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=100): Hit Rate=0.7508, MRR=0.3913


  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=200): Hit Rate=0.7508, MRR=0.3913


## Results

| Method | Hit Rate | MRR |
|---|---|---|
| Keyword | 0.556 | 0.338 |
| Vector | 0.633 | 0.404 |
| Hybrid (k=60, RRF default) | 0.751 | 0.391 |
| **Hybrid (k=1)** | **0.751** | **0.421** |

Hybrid search wins on Hit Rate outright, and with `k=1` it also beats vector search on MRR — the best of both metrics. **Hybrid search with `k=1` is the winner**, wired into `coffee_assistant/retrieval.py` (`hybrid_search`'s default) and used in production by `rag.py`'s `search()`.

In [12]:
# Final pick: hybrid search, k=1 — this is retrieval.hybrid_search()'s actual default,
# and what rag.py's search() calls in production.
evaluate(ground_truth, lambda q: retrieval.hybrid_search(q['question'], k=1))

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.7507936507936508, 'mrr': 0.42130322499370204}